In [3]:
%%capture

!pip uninstall -y \
    transformers trl unsloth unsloth_zoo \
    peft accelerate bitsandbytes xformers

!pip install -U pip

!pip install \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

!pip install \
    transformers==4.56.0 \
    trl>=0.9.0 \
    peft \
    accelerate \
    bitsandbytes \
    xformers

In [9]:
# Necessary imports
from unsloth import FastLanguageModel
import torch

# Parameters for loading the quanitized version of the model
max_seq_length = 2048
dtyp = None
load_in_4bit = True

# Set model and tokenizer parameters
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    torch_dtype = dtyp,
    load_in_4bit = load_in_4bit,
    device_map = "auto"
)

# Inject LoRA adapters so we can freeze the other layers and only train a subset of the layers
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

==((====))==  Unsloth 2026.5.7: Fast Llama patching. Transformers: 4.56.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth 2026.5.7 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


In [12]:
# Format actual dataset
from datasets import Dataset
from datasets import load_dataset

# Defining the formatting template
instruction_prompt = """Below is a statement. Analyze its sentiment and output a strict decimal score between 0.0 (extremely negative/furious) and 1.0 (extremely positive/thrilled). Do not include any other text or explanation.

### Statement:
{}

### Score:
{}"""

EOS_TOKEN = tokenizer.eos_token
# The parameter 'examples' is a dictionary
def formatting_prompts_func(examples):
  inputs = examples["input"]
  outputs = examples["output"]
  texts = []
  for input, output in zip(inputs, outputs):
    texts.append(instruction_prompt.format(str(input), str(output)) + EOS_TOKEN)

  return {"text": texts}

dataset = load_dataset("csv", data_files="gpt_sentiment_data.csv", split="train")

# We apply the mapping function to actually build the "text" column in memory
dataset = dataset.map(formatting_prompts_func, batched=True)
print("Model loaded and dataset formatted successfully!")

Model loaded and dataset formatted successfully!


In [13]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Initialize the Trainer
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Run the training loop
trainer_stats = trainer.train()
print(f"Training complete! Final Loss: {trainer_stats.metrics['train_loss']}")

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/102 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 102 | Num Epochs = 3 | Total steps = 39
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
1,0.401100
2,0.358900
3,0.363000
4,0.380400
5,0.433500
6,0.437400
7,0.392600
8,0.397400
9,0.369100
10,0.369800


wandb: WARNING URL not available in offline run


Training complete! Final Loss: 0.26708905475261885


In [16]:
# Inference mode
FastLanguageModel.for_inference(model)

prompt = "I'm racist"

inputs = tokenizer(
    [
        instruction_prompt.format(prompt, ""),
    ],
    return_tensors = "pt",
).to("cuda")

# Generate prediction
outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
prediction = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
print(prediction)


Below is a statement. Analyze its sentiment and output a strict decimal score between 0.0 (extremely negative/furious) and 1.0 (extremely positive/thrilled). Do not include any other text or explanation.

### Statement:
I'm racist

### Score:
0.01


In [17]:
# Saving the model locally with 4 bit quantization
model.save_pretrained("sentiment_check_model")
tokenizer.save_pretrained("sentiment_check_model")

('sentiment_check_model/tokenizer_config.json',
 'sentiment_check_model/special_tokens_map.json',
 'sentiment_check_model/chat_template.jinja',
 'sentiment_check_model/tokenizer.json')

In [18]:
!zip -r vibe_check_model.zip sentiment_check_model

updating: sentiment_check_model/ (stored 0%)
updating: sentiment_check_model/tokenizer_config.json (deflated 96%)
updating: sentiment_check_model/special_tokens_map.json (deflated 71%)
updating: sentiment_check_model/adapter_model.safetensors (deflated 8%)
updating: sentiment_check_model/adapter_config.json (deflated 59%)
updating: sentiment_check_model/README.md (deflated 65%)
updating: sentiment_check_model/chat_template.jinja (deflated 71%)
updating: sentiment_check_model/tokenizer.json (deflated 85%)


In [19]:
from google.colab import files
files.download("vibe_check_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
model.save_pretrained_merged(
    "merged_model",
    tokenizer,
    save_method="merged_16bit",
)

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:52<00:00, 112.53s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:19<00:00, 79.22s/it]


Unsloth: Merge process complete. Saved to `/content/merged_model`


In [21]:
model.save_pretrained_gguf(
    "gguf_model",
    tokenizer,
    quantization_method = "q4_k_m",
)

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [03:26<00:00, 206.59s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:36<00:00, 96.41s/it]


Unsloth: Merge process complete. Saved to `/content/gguf_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['gguf_model_gguf/Llama-3.2-1B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files c

{'save_directory': 'gguf_model',
 'gguf_directory': 'gguf_model_gguf',
 'gguf_files': ['gguf_model_gguf/Llama-3.2-1B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': 'gguf_model_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}